In [12]:
!pip install pyspark==4.0.1 grpcio grpcio-status googleapis-common-protos pyarrow

In [13]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .remote("sc://spark-connect.kubeflow-user-example-com.svc.cluster.local:15002") \
    .getOrCreate()

# Test it
df = spark.range(100)
df.show()
print("Spark version:", spark.version)

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
| 10|
| 11|
| 12|
| 13|
| 14|
| 15|
| 16|
| 17|
| 18|
| 19|
+---+
only showing top 20 rows
Spark version: 4.0.1


In [4]:
from pyspark.sql.functions import col, rand, when

df = spark.range(1000) \
    .withColumn("value", rand()) \
    .withColumn("category", when(col("id") % 3 == 0, "A")
                            .when(col("id") % 3 == 1, "B")
                            .otherwise("C"))

df.groupBy("category") \
  .agg({"value": "avg", "id": "count"}) \
  .orderBy("category") \
  .show()

+--------+-------------------+---------+
|category|         avg(value)|count(id)|
+--------+-------------------+---------+
|       A|0.46644451597437736|      334|
|       B| 0.5254982114853131|      333|
|       C| 0.5001599778559895|      333|
+--------+-------------------+---------+



In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand, col
import time

spark = SparkSession.builder \
    .remote("sc://spark-connect.kubeflow-user-example-com.svc.cluster.local:15002") \
    .getOrCreate()

# Generate large dataset and do heavy aggregations to keep executors busy
df = spark.range(0, 100_000_000, numPartitions=100) \
    .withColumn("rand1", rand()) \
    .withColumn("rand2", rand()) \
    .withColumn("rand3", rand()) \
    .withColumn("bucket", (col("id") % 1000).cast("string"))

# Force multiple shuffle stages to keep it running ~1 min
result = df.groupBy("bucket") \
    .agg({"rand1": "sum", "rand2": "avg", "rand3": "max"}) \
    .orderBy("bucket") \
    .count()

print(f"Done! Processed {result} buckets")

SparkException: Job aborted due to stage failure: ShuffleMapStage 3 (Spark Connect - session_id: "a2031e6b-6921-4e04-b51c-d0971443094e"
plan {
  root {
    common {
      plan_id: 32
    }
    aggregate {
     ...) has failed the maximum allowable number of times: 4. Most recent failure reason:
org.apache.spark.shuffle.FetchFailedException
	at org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:439)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1256)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:983)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:87)
	at org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)
	at scala.collection.Iterator$$anon$10.nextCur(Iterator.scala:594)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:608)
	at scala.collection.Iterator$$a